# QICK Jodrell Bank Survey Notebook
### RFSoC 4x2 — Validated for QICK 0.2.388

**Run cells top to bottom. Every cell prints PASS, WARN, or FAIL.**

All QICK 0.2.388 API fixes are already applied:
- `soc.config = soc.get_cfg()` patched after load
- DDR4 uses `ddr4=True` trigger flag
- `capture_spectrum()` uses the confirmed working sequence
- USB stick save path auto-detected

---

## How to run at Jodrell
1. **Cell 1–5**: Environment checks — all must show PASS
2. **Cell 6**: Connect hardware, find USB mount point
3. **Cell 7**: Health check with LNA connected — check RMS and clipping
4. **Cell 8**: Configure observation parameters
5. **Cell 9**: Run observation — prints live status every spectrum

**If anything FAILs**, read the message — it will tell you exactly what to fix.

## Cell 1 — Python environment check

In [41]:
import sys, os, time, datetime, subprocess
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

REQUIRED = ['numpy', 'matplotlib', 'qick']
missing  = []
for pkg in REQUIRED:
    try:
        __import__(pkg)
        print(f'   {pkg} importable')
    except ImportError:
        print(f'   MISSING: {pkg}')
        missing.append(pkg)

print(f'\nPython     : {sys.version.split()[0]}')
print(f'NumPy      : {np.__version__}')
print(f'Matplotlib : {matplotlib.__version__}')

if missing:
    print(f'\n FAIL — install missing packages: pip install {" ".join(missing)}')
else:
    print('\n PASS — all packages available')

   numpy importable
   matplotlib importable
   qick importable

Python     : 3.10.4
NumPy      : 1.21.5
Matplotlib : 3.5.1

 PASS — all packages available


## Cell 2 — PYNQ version (must be 3.0.x)

In [42]:
import pkg_resources
try:
    v = pkg_resources.get_distribution('pynq').version
    major, minor = int(v.split('.')[0]), int(v.split('.')[1])
    print(f'  PYNQ version: {v}')
    if major == 3 and minor == 0:
        print('  PASS — PYNQ 3.0.x confirmed (QICK compatible)')
    else:
        print('  FAIL — QICK 0.2.388 requires PYNQ 3.0.x')
        print('  Fix: reflash SD card with PYNQ 3.0.1 image')
except Exception as e:
    print(f'  FAIL — {e}')

  PYNQ version: 3.0.1
  PASS — PYNQ 3.0.x confirmed (QICK compatible)


## Cell 3 — QICK version (must be 0.2.x)

In [43]:
try:
    import qick
    v = pkg_resources.get_distribution('qick').version
    print(f'  QICK version: {v}')
    if v.startswith('0.2'):
        print('  PASS — QICK 0.2.x confirmed')
    else:
        print(f'  WARN — untested version {v} (validated on 0.2.388)')
except Exception as e:
    print(f'  FAIL — {e}')

  QICK version: 0.2.388
  PASS — QICK 0.2.x confirmed


## Cell 4 — Bitstream files present

In [44]:
BIT_FILE = '/home/xilinx/qick_repo/qick_lib/qick/qick_4x2.bit'
HWH_FILE = '/home/xilinx/qick_repo/qick_lib/qick/qick_4x2.hwh'

ok = True
for f in [BIT_FILE, HWH_FILE]:
    if os.path.exists(f):
        print(f'  Found: {os.path.basename(f)} ({os.path.getsize(f)/1e6:.1f} MB)')
    else:
        print(f'  MISSING: {f}')
        ok = False

if ok:
    print('\n PASS — bitstream files confirmed')
else:
    print('\n FAIL — copy bitstream to board:')
    print('  scp -r ~/qick_repo xilinx@192.168.3.1:/home/xilinx/')

  Found: qick_4x2.bit (34.4 MB)
  Found: qick_4x2.hwh (1.8 MB)

 PASS — bitstream files confirmed


## Cell 5 — Load QICK overlay
**Takes ~60 seconds. The FPGA is being programmed. Do not interrupt.**

In [45]:
from qick import QickSoc
from qick.averager_program import AveragerProgram

print('[LOAD] Programming FPGA — please wait ~60 seconds...')
t0 = time.time()
try:
    soc = QickSoc(bitfile=BIT_FILE)
    # ── CRITICAL API FIX: patch soc.config for QICK 0.2.388 ──────────────────
    # soc.config['key'] does not work in this version.
    # This one line makes the whole notebook work correctly.
    soc.config = soc.get_cfg()
    print(f'\n[LOAD] Done in {time.time()-t0:.1f}s')
    print('\n PASS — overlay loaded')
    print('\n--- Hardware configuration ---')
    print(soc)
except Exception as e:
    soc = None
    print(f'\n FAIL — {e}')
    print('  Check: is PYNQ 3.0.1 running? Is the bitstream file present?')

[LOAD] Programming FPGA — please wait ~60 seconds...

[LOAD] Done in 4.2s

 PASS — overlay loaded

--- Hardware configuration ---
QICK running on RFSoC4x2, software version 0.2.388

Firmware configuration (built Wed Sep  6 18:49:29 2023):

	Global clocks (MHz): tProc dispatcher timing 409.600, RF reference 491.520
	Groups of related clocks: [tProc clock, DAC tile 0], [DAC tile 2], [ADC tile 0]

	2 signal generator channels:
	0:	axis_signal_gen_v6 - fs=9830.400 Msps, fabric=614.400 MHz
		envelope memory: 65536 complex samples (6.667 us)
		32-bit DDS, range=9830.400 MHz
		DAC tile 0, blk 0 is DAC_B
	1:	axis_signal_gen_v6 - fs=9830.400 Msps, fabric=614.400 MHz
		envelope memory: 65536 complex samples (6.667 us)
		32-bit DDS, range=9830.400 MHz
		DAC tile 2, blk 0 is DAC_A

	2 readout channels:
	0:	axis_readout_v2 - configured by PYNQ
		fs=4423.680 Msps, decimated=552.960 MHz, 32-bit DDS, range=4423.680 MHz
		axis_avg_buffer v1.0 (no edge counter, no weights)
		memory 16384 accumulated, 10

## Cell 6 — USB stick detection and hardware configuration

**Before running this cell:**
- Plug the USB stick into the RFSoC USB port
- Connect: Antenna → LNA → ADC_D
- Remove any loopback cable

In [46]:
if soc is None:
    print('FAIL — soc not loaded. Re-run Cell 5.')
else:
    # ── Storage mode — change this one line ──────────────────────────────
    # Set to 'usb'   to auto-detect USB stick
    # Set to 'local' to save to board disk
    STORAGE_MODE = 'local'

    if STORAGE_MODE == 'usb':
        result = subprocess.run(
            ['lsblk', '-o', 'NAME,MOUNTPOINT,SIZE,FSTYPE'],
            capture_output=True, text=True)
        print('Mounted drives:')
        print(result.stdout)
        USB_PATH = None
        for line in result.stdout.split('\n'):
            for mount in ['/media/', '/mnt/']:
                if mount in line:
                    parts = line.split()
                    for p in parts:
                        if p.startswith(mount):
                            USB_PATH = p + '/'
                            break
        if USB_PATH and os.path.exists(USB_PATH):
            SAVE_PATH = USB_PATH
            print(f'  USB detected: {SAVE_PATH}')
            print('  PASS — saving to USB stick')
        else:
            print('  WARN — USB not found, falling back to local disk')
            SAVE_PATH = '/home/xilinx/jupyter_notebooks/spectrum-analyzer/'
            os.makedirs(SAVE_PATH, exist_ok=True)

    elif STORAGE_MODE == 'local':
        SAVE_PATH = '/home/xilinx/jupyter_notebooks/spectrum-analyzer/'
        os.makedirs(SAVE_PATH, exist_ok=True)
        free_gb = os.statvfs(SAVE_PATH).f_bavail * \
                  os.statvfs(SAVE_PATH).f_frsize / 1e9
        print(f'  Saving locally to: {SAVE_PATH}')
        print(f'  Free disk space  : {free_gb:.1f} GB')
        if free_gb < 1.0:
            print('  WARN — less than 1 GB free, consider using USB')
        else:
            print('  PASS — sufficient disk space')

    print(f'\n  Active save path: {SAVE_PATH}')

    # ── Hardware configuration ────────────────────────────────────────────
    cfg         = soc.get_cfg()
    ADC_CH      = 0
    FS_MHZ      = cfg['readouts'][ADC_CH]['fs']
    NYQUIST_MHZ = FS_MHZ / 2.0
    FFT_SIZE    = 1_048_576
    NT          = (FFT_SIZE // 256) + 10
    N_BINS      = FFT_SIZE // 2
    DF_KHZ      = FS_MHZ * 1e3 / FFT_SIZE

    freq_axis_mhz = np.linspace(0, NYQUIST_MHZ, N_BINS,
                                 endpoint=False).astype(np.float32)
    window        = np.hanning(FFT_SIZE).astype(np.float32)
    window_power  = float(np.sum(window**2))

    # ── DDR4 trigger program ──────────────────────────────────────────────
    class DDR4CaptureProgram(AveragerProgram):
        def initialize(self):
            self.declare_readout(ch=ADC_CH, length=1000, freq=0, gen_ch=None)
            self.synci(200)
        def body(self):
            self.trigger(adcs=[ADC_CH], ddr4=True, adc_trig_offset=100)
            self.wait_all()
            self.sync_all(self.us2cycles(1.0))

    _ddr4_prog = DDR4CaptureProgram(soc, {
        'ro_ch':ADC_CH, 'readout_length':1000,
        'adc_trig_offset':100, 'soft_avgs':1,
        'reps':1, 'relax_delay':1.0
    })

    # ── Capture function ──────────────────────────────────────────────────
    def capture_spectrum():
        _cfg = soc.get_cfg()
        soc.ddr4_buf.set_switch(_cfg['readouts'][ADC_CH]['avgbuf_fullpath'])
        soc.clear_ddr4()
        soc.ddr4_buf.arm(nt=NT)
        _ddr4_prog.acquire(soc, load_pulses=False, progress=False)
        raw    = soc.ddr4_buf.get_mem(nt=NT)
        i_data = (raw[:,0] if raw.ndim==2 else raw).astype(np.float32)
        i_data = i_data[:FFT_SIZE] if len(i_data) >= FFT_SIZE \
                 else np.pad(i_data, (0, FFT_SIZE - len(i_data)))
        clip   = float(np.mean(np.abs(i_data) > 0.95 * 32767))
        spec   = np.abs(np.fft.rfft(i_data * window, n=FFT_SIZE))**2 \
                 / window_power
        return 10 * np.log10(spec[:N_BINS] + 1e-30).astype(np.float32), clip

    print(f'  ADC channel    : {ADC_CH} (ADC_D)')
    print(f'  Sample rate    : {FS_MHZ:.3f} MHz')
    print(f'  Nyquist        : {NYQUIST_MHZ:.3f} MHz')
    print(f'  FFT size       : {FFT_SIZE:,}')
    print(f'  Resolution     : {DF_KHZ:.2f} kHz/bin')
    print(f'  DDR4 transfers : {NT}')
    print('\n PASS — hardware configuration complete')

  Saving locally to: /home/xilinx/jupyter_notebooks/spectrum-analyzer/
  Free disk space  : 5.1 GB
  PASS — sufficient disk space

  Active save path: /home/xilinx/jupyter_notebooks/spectrum-analyzer/
  ADC channel    : 0 (ADC_D)
  Sample rate    : 4423.680 MHz
  Nyquist        : 2211.840 MHz
  FFT size       : 1,048,576
  Resolution     : 4.22 kHz/bin
  DDR4 transfers : 4106

 PASS — hardware configuration complete


## Cell 7 — Signal health check

**Run this immediately after connecting the LNA and antenna.**

What to look for:
- RMS between 5 and 500 ADU → healthy signal
- Clipping 0.00% → no attenuation needed
- Clipping > 1% → run `soc.set_adc_attenuator('00', 20)` then re-run this cell

In [47]:
if soc is None:
    print('FAIL — soc not loaded. Re-run Cell 5.')
else:
    print('Running signal health check...')
    print('(Capturing 3 successive DDR4 buffers to verify live data)')

    results = []
    for k in range(3):
        _cfg = soc.get_cfg()
        soc.ddr4_buf.set_switch(_cfg['readouts'][ADC_CH]['avgbuf_fullpath'])
        soc.clear_ddr4()
        soc.ddr4_buf.arm(nt=256)
        _ddr4_prog.acquire(soc, load_pulses=False, progress=False)
        raw    = soc.ddr4_buf.get_mem(nt=256)
        i_data = (raw[:,0] if raw.ndim==2 else raw).astype(np.float32)
        rms    = float(np.sqrt(np.mean(i_data**2)))
        clip   = float(np.mean(np.abs(i_data) > 0.95 * 32767)) * 100
        results.append(i_data[:500].copy())
        print(f'  Capture {k+1}: RMS={rms:.2f} ADU  clip={clip:.2f}%  '
              f'first 3 values={i_data[:3]}')

    is_live      = not np.allclose(results[0], results[1])
    rms_final    = float(np.sqrt(np.mean(results[0]**2)))
    clip_final   = float(np.mean(np.abs(results[0]) > 0.95 * 32767)) * 100

    print()
    print(f'  Data is live (changes between captures) : {is_live}')
    print(f'  RMS                                     : {rms_final:.2f} ADU')
    print(f'  Clipping                                : {clip_final:.2f}%')
    print(f'  Current attenuator                      : '
          f'{soc.get_adc_attenuator("00")} dB')

    print()
    if not is_live:
        print('  FAIL — DDR4 buffer returning stale data')
        print('  Fix: restart the kernel and re-run from Cell 1')
    elif clip_final > 5.0:
        print('  WARN — high clipping detected')
        print('  Fix: run soc.set_adc_attenuator("00", 20) then re-run this cell')
        print('       If still clipping at 20 dB, try 27 dB')
    elif rms_final < 0.5:
        print('  WARN — signal very low, check antenna and LNA connections')
    else:
        spec_test, _ = capture_spectrum()
        print(f'  Spectrum peak  : {float(spec_test.max()):.1f} dB')
        print(f'  Spectrum median: {float(np.median(spec_test)):.1f} dB')
        print()
        print('  PASS — signal healthy, ready to observe')

Running signal health check...
(Capturing 3 successive DDR4 buffers to verify live data)
  Capture 1: RMS=129.97 ADU  clip=0.00%  first 3 values=[ -8. -44. -55.]
  Capture 2: RMS=70.64 ADU  clip=0.00%  first 3 values=[24.  0.  8.]
  Capture 3: RMS=62.99 ADU  clip=0.00%  first 3 values=[-8. 63. 21.]

  Data is live (changes between captures) : True
  RMS                                     : 59.46 ADU
  Clipping                                : 0.00%
  Current attenuator                      : 0.0 dB

  Spectrum peak  : 77.2 dB
  Spectrum median: 24.9 dB

  PASS — signal healthy, ready to observe


## Cell 8 — Observation configuration

**Edit the parameters below then run this cell.**

For the Jodrell paper measurement use the default values below.

In [48]:
# ── Edit these parameters ────────────────────────────────────────────────────
OBSERVATION_HOURS  = 0.1    # Duration. 0.5h = 30 min. Increase for longer runs.
N_AVG              = 5      # Frames averaged per spectrum. Higher = smoother.
SAVE_INTERVAL_MINS = 5      # Save intermediate PNG every N minutes.
MAX_WATERFALL_ROWS = 200    # Maximum waterfall rows kept in memory.
ADC_ATTENUATION_DB = 0      # Set to 20 or 27 if Cell 7 showed clipping.
WINDOW_FUNCTION    = 'hann'

# ── RFI band markers ──────────────────────────────────────────────────────────
RFI = {
    'FM\n(88-108)'   : (88,   108,  '#aaffaa'),
    'DAB\n(174-240)' : (174,  240,  '#aaffaa'),
    '4G\n(700-960)'  : (700,  960,  '#ffaaff'),
    'GPS\n(1575)'    : (1570, 1580, '#aaaaff'),
}
WH_FM  = [88.4, 90.4, 94.6, 96.0, 97.6, 99.4, 102.0, 103.0]
WH_DAB = [209.936, 222.064]

# ── Apply attenuation ─────────────────────────────────────────────────────────
if ADC_ATTENUATION_DB > 0:
    soc.set_adc_attenuator('00', ADC_ATTENUATION_DB)
    print(f'  Attenuator set to {soc.get_adc_attenuator("00")} dB')
else:
    print(f'  Attenuator: 0 dB (no attenuation)')

OBSERVATION_SECONDS = OBSERVATION_HOURS * 3600

print(f'\n  Duration     : {OBSERVATION_HOURS}h ({OBSERVATION_HOURS*60:.0f} minutes)')
print(f'  Resolution   : {DF_KHZ:.2f} kHz/bin')
print(f'  Averaging    : {N_AVG} frames/spectrum')
print(f'  Save every   : {SAVE_INTERVAL_MINS} minutes')
print(f'  Save path    : {SAVE_PATH}')
print(f'  Nyquist      : {NYQUIST_MHZ:.1f} MHz')
print()
print('  READY — run Cell 9 to start the observation')

  Attenuator: 0 dB (no attenuation)

  Duration     : 0.1h (6 minutes)
  Resolution   : 4.22 kHz/bin
  Averaging    : 5 frames/spectrum
  Save every   : 5 minutes
  Save path    : /home/xilinx/jupyter_notebooks/spectrum-analyzer/
  Nyquist      : 2211.8 MHz

  READY — run Cell 9 to start the observation


## Cell 9 — Run observation

**This cell runs the full observation.**

- Watch the live status lines — `peak` should be well above `noise`
- `clip 0.00%` means the ADC is not saturating — good
- Files save automatically every 5 minutes to the USB stick
- Press ⬛ Stop or Kernel → Interrupt to stop early and save

**If you see errors:** the cell will retry 5 times before aborting. Each error message tells you exactly what went wrong.

In [49]:
# ── Plot function ────────────────────────────────────────────────────────────
def save_plot(ts, wf, mh, ms, ns, elapsed, clip_pct):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 10))
    fig.patch.set_facecolor('#0d0d0d')

    ax1.set_facecolor('#0d0d0d')
    ax1.plot(freq_axis_mhz, ms, color='#00e5ff', lw=0.4, alpha=0.8, label='Mean')
    ax1.plot(freq_axis_mhz, mh, color='#ff6b35', lw=0.5, label='Max hold')

    for lbl, (f0, f1, col) in RFI.items():
        if f0 < float(freq_axis_mhz[-1]):
            ax1.axvspan(f0, min(f1, float(freq_axis_mhz[-1])),
                        alpha=0.08, color=col)
            ax1.text((f0 + min(f1, float(freq_axis_mhz[-1]))) / 2,
                     mh.max() + 1, lbl,
                     color=col, fontsize=5, ha='center', va='bottom')

    for f in WH_FM:
        ax1.axvline(f, color='#88ff88', lw=0.5, ls='--', alpha=0.5)
    ax1.axvline(WH_FM[0], color='#88ff88', lw=0.5, ls='--',
                alpha=0.5, label='Winterhill FM')
    for f in WH_DAB:
        ax1.axvline(f, color='#ffff88', lw=0.5, ls=':', alpha=0.5)
    ax1.axvline(WH_DAB[0], color='#ffff88', lw=0.5, ls=':',
                alpha=0.5, label='Winterhill DAB')

    ax1.set_xlim(0, NYQUIST_MHZ)
    ax1.set_ylabel('Power (dB)', color='white', fontsize=11)
    ax1.set_title(
        f'QICK DDR4 RFI Survey | RFSoC 4x2 | UTC {ts} | '
        f'{elapsed/3600:.2f}h | {ns} spectra | '
        f'{DF_KHZ:.2f} kHz/bin | Clip: {clip_pct:.2f}% | '
        f'Jodrell Bank — Yagi + LNA',
        color='white', fontsize=9)
    ax1.tick_params(colors='white')
    ax1.spines[:].set_color('#333333')
    ax1.grid(True, color='#1e1e1e', lw=0.3)
    ax1.xaxis.set_major_locator(ticker.MultipleLocator(200))
    ax1.legend(facecolor='#1a1a1a', edgecolor='#444',
               labelcolor='white', fontsize=7, loc='upper right')

    ax2.set_facecolor('#0d0d0d')
    if len(wf) > 1:
        im = ax2.imshow(np.array(wf), aspect='auto',
                        extent=[0, NYQUIST_MHZ, elapsed/3600, 0],
                        cmap='inferno',
                        vmin=np.percentile(wf, 2),
                        vmax=np.percentile(wf, 98))
        cb = plt.colorbar(im, ax=ax2, label='Power (dB)', pad=0.01)
        cb.ax.yaxis.label.set_color('white')
        cb.ax.tick_params(colors='white')

    ax2.set_xlabel('Frequency (MHz)', color='white', fontsize=11)
    ax2.set_ylabel('Time (hours)',    color='white', fontsize=11)
    ax2.set_title(
        f'Waterfall ({ns} rows | {DF_KHZ:.2f} kHz/bin | {WINDOW_FUNCTION})',
        color='white', fontsize=9)
    ax2.tick_params(colors='white')
    ax2.xaxis.set_major_locator(ticker.MultipleLocator(200))

    plt.tight_layout()
    out = f'{SAVE_PATH}qick_jodrell_{ts}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
    plt.close(fig)
    return out

# ── Acquisition loop ──────────────────────────────────────────────────────────
ts           = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
max_hold     = np.full(N_BINS, -200.0, dtype=np.float32)
waterfall    = []
frame_acc    = np.zeros(N_BINS, dtype=np.float64)
n_frames_acc = n_spectra = n_clips = n_total = consecutive_errors = 0
t_start      = t_last_save = time.time()

print(f'[OBS] Started UTC {ts}')
print(f'[OBS] {OBSERVATION_HOURS}h | {DF_KHZ:.2f} kHz/bin | '
      f'{N_AVG} frames/spectrum | {WINDOW_FUNCTION} window')
print(f'[OBS] Saving to: {SAVE_PATH}')
print(f'[OBS] Press Stop or Kernel → Interrupt to stop early\n')

try:
    while (time.time() - t_start) < OBSERVATION_SECONDS:

        # ── Capture one spectrum ──────────────────────────────────────────────
        try:
            spec_db, clip_frac = capture_spectrum()
            consecutive_errors = 0
        except Exception as e:
            consecutive_errors += 1
            print(f'  [WARN] Capture error ({consecutive_errors}/5): {e}')
            if consecutive_errors >= 5:
                print('  [STOP] Too many consecutive errors — aborting')
                print('  Fix: restart kernel and re-run from Cell 1')
                break
            time.sleep(2)
            continue

        n_total += 1

        # ── Clipping warning ──────────────────────────────────────────────────
        if clip_frac > 0.01:
            n_clips += 1
            if n_clips <= 3 or n_clips % 20 == 0:
                print(f'  [CLIP] {clip_frac*100:.1f}% clipping — '
                      f'run soc.set_adc_attenuator("00", 20) to fix')

        # ── Accumulate frames ─────────────────────────────────────────────────
        frame_acc    += spec_db.astype(np.float64)
        n_frames_acc += 1

        if n_frames_acc >= N_AVG:
            averaged     = (frame_acc / n_frames_acc).astype(np.float32)
            max_hold     = np.maximum(max_hold, averaged)
            if len(waterfall) >= MAX_WATERFALL_ROWS:
                waterfall.pop(0)
            waterfall.append(averaged.copy())
            n_spectra   += 1
            frame_acc[:] = 0.0
            n_frames_acc = 0

            elapsed  = time.time() - t_start
            clip_pct = n_clips / n_total * 100
            print(f'  #{n_spectra:5d} | frames {n_total:6d} | '
                  f'{elapsed/3600:5.2f}h / {OBSERVATION_HOURS}h | '
                  f'clip {clip_pct:.2f}% | '
                  f'peak {averaged.max():.1f} dB | '
                  f'noise {np.median(averaged):.1f} dB')

        # ── Periodic save ─────────────────────────────────────────────────────
        if (time.time() - t_last_save) >= SAVE_INTERVAL_MINS * 60 \
                and n_spectra > 0:
            elapsed  = time.time() - t_start
            clip_pct = n_clips / n_total * 100
            wf_arr   = np.array(waterfall, dtype=np.float32)
            ms       = np.mean(wf_arr, axis=0)
            np.save(f'{SAVE_PATH}qick_jodrell_maxhold_{ts}.npy',   max_hold)
            np.save(f'{SAVE_PATH}qick_jodrell_waterfall_{ts}.npy', wf_arr)
            np.save(f'{SAVE_PATH}qick_jodrell_freqaxis_{ts}.npy',  freq_axis_mhz)
            pp = save_plot(ts, waterfall, max_hold, ms,
                           n_spectra, elapsed, clip_pct)
            snap = datetime.datetime.utcnow().strftime('%H:%M:%S')
            print(f'\n  [SAVE] {snap} — {n_spectra} spectra saved | '
                  f'{os.path.basename(pp)}\n')
            t_last_save = time.time()

except KeyboardInterrupt:
    print('\n[STOP] Interrupted — saving final result...')

# ── Final save ────────────────────────────────────────────────────────────────
elapsed_total = time.time() - t_start
clip_pct      = n_clips / n_total * 100 if n_total > 0 else 0
wf_arr        = np.array(waterfall, dtype=np.float32)
mean_spec     = np.mean(wf_arr, axis=0) if len(wf_arr) > 0 else max_hold

np.save(f'{SAVE_PATH}qick_jodrell_maxhold_{ts}.npy',   max_hold)
np.save(f'{SAVE_PATH}qick_jodrell_waterfall_{ts}.npy', wf_arr)
np.save(f'{SAVE_PATH}qick_jodrell_freqaxis_{ts}.npy',  freq_axis_mhz)
pp = save_plot(ts, waterfall, max_hold, mean_spec,
               n_spectra, elapsed_total, clip_pct)

print(f'\n[DONE]')
print(f'  Duration   : {elapsed_total/3600:.2f}h')
print(f'  Frames     : {n_total}')
print(f'  Spectra    : {n_spectra}')
print(f'  Clipping   : {clip_pct:.2f}%')
print(f'  Resolution : {DF_KHZ:.2f} kHz/bin')
print(f'  Saved to   : {SAVE_PATH}')
print(f'\nDownload with:')
print(f'  scp xilinx@192.168.3.1:{pp} .')
print(f'  scp xilinx@192.168.3.1:{SAVE_PATH}qick_jodrell_*_{ts}.npy .')

[OBS] Started UTC 20250608_204151
[OBS] 0.1h | 4.22 kHz/bin | 5 frames/spectrum | hann window
[OBS] Saving to: /home/xilinx/jupyter_notebooks/spectrum-analyzer/
[OBS] Press Stop or Kernel → Interrupt to stop early

  #    1 | frames      5 |  0.01h / 0.1h | clip 0.00% | peak 72.9 dB | noise 24.3 dB
  #    2 | frames     10 |  0.01h / 0.1h | clip 0.00% | peak 73.3 dB | noise 24.3 dB
  #    3 | frames     15 |  0.02h / 0.1h | clip 0.00% | peak 71.2 dB | noise 24.2 dB
  #    4 | frames     20 |  0.03h / 0.1h | clip 0.00% | peak 72.4 dB | noise 24.3 dB
  #    5 | frames     25 |  0.04h / 0.1h | clip 0.00% | peak 71.2 dB | noise 24.3 dB
  #    6 | frames     30 |  0.04h / 0.1h | clip 0.00% | peak 72.2 dB | noise 24.3 dB
  #    7 | frames     35 |  0.05h / 0.1h | clip 0.00% | peak 73.9 dB | noise 24.3 dB
  #    8 | frames     40 |  0.06h / 0.1h | clip 0.00% | peak 73.2 dB | noise 24.4 dB
  #    9 | frames     45 |  0.06h / 0.1h | clip 0.00% | peak 71.9 dB | noise 24.4 dB
  #   10 | frames   